# B03 · A ball detector for broadcast basketball (Google Colab)

**hoop-reid-tracker**, ball chapter, step B03.

### Why
SportsMOT has no ball labels, and the COCO detector finds the ball in only 8-43 % of the frames on three of four
arenas, with confident false alarms in the stands (B01). So the ball was labelled by hand: one click on its centre in
one frame out of ten of the 15 **train** clips (`scripts/ball/b02_label_ball.py`); the box size comes from the players
around it (`hooptrack.ball.labels.ball_diameter`). `scripts/ball/b03_build_ball_dataset.py` turned the clicks into a
YOLO dataset: 1,245 full frames, ~84 % with the ball, the others negatives; the last clip of each train game is the
`val` split (best epoch). The SportsMOT validation clips are never used here: they are the test (B04, on the laptop).

### Model
A **separate** detector, not the player detector: the ball needs full resolution (at 640 px it is 5-10 px wide) and
a single class; the player detector, its cache and every result of the tracking chapter stay as they are. YOLO26s,
COCO-pretrained (it already knows `sports ball`), fine-tuned at **1280 px**.

### Steps
1. Upload `data/ball_det.zip` to `MyDrive/hooptrack/`.
2. Runtime → Change runtime type → **T4 GPU**, then Run all.
3. Download `MyDrive/hooptrack/runs/ball_yolo26s_1280/weights/best.pt` to `weights/ball_yolo26s_1280.pt` on the laptop.


## 1 · GPU and libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "ultralytics>=8.4"

## 2 · Configuration

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")
    WORK = Path("/content")
else:
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

ZIP = DRIVE_DIR / "ball_det.zip"
RUNS = (DRIVE_DIR / "runs").resolve()       # absolute: ultralytics puts relative paths under runs/detect/
MODEL = os.environ.get("HOOPTRACK_MODEL", "yolo26s.pt")
IMGSZ = int(os.environ.get("HOOPTRACK_IMGSZ", 1280))
EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 100))
PATIENCE, BATCH = 30, int(os.environ.get("HOOPTRACK_BATCH", 8))
NAME = f"ball_{Path(MODEL).stem}_{IMGSZ}"
WORK.mkdir(parents=True, exist_ok=True)
print(NAME, "->", RUNS / NAME)

## 3 · Data

In [ ]:
import zipfile
DATA_DIR = WORK / "ball_det"
if not (DATA_DIR / "data.yaml").is_file():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)
(DATA_DIR / "data.yaml").write_text(f"path: {DATA_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n")
for s in ("train", "val"):
    labs = list((DATA_DIR / "labels" / s).glob("*.txt"))
    print(s, len(labs), "images,", sum(1 for p in labs if p.read_text().strip()), "with the ball")

### What the detector learns (green: the box made from the click)

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
random.seed(0)
imgs = [p for p in sorted((DATA_DIR / "images" / "train").glob("*.jpg"))
        if (DATA_DIR / "labels" / "train" / (p.stem + ".txt")).read_text().strip()]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, p in zip(axes.ravel(), random.sample(imgs, 12)):
    img = cv2.imread(str(p)); H, W = img.shape[:2]
    _, cx, cy, w, h = map(float, (DATA_DIR / "labels" / "train" / (p.stem + ".txt")).read_text().split())
    x1, y1, x2, y2 = int((cx - w / 2) * W), int((cy - h / 2) * H), int((cx + w / 2) * W), int((cy + h / 2) * H)
    crop = img[max(0, y1 - 60):y2 + 60, max(0, x1 - 60):x2 + 60].copy()
    cv2.rectangle(crop, (min(60, x1), min(60, y1)), (min(60, x1) + x2 - x1, min(60, y1) + y2 - y1), (0, 255, 0), 1)
    ax.imshow(crop[..., ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Training (resumes from `last.pt` if the session was cut)

In [ ]:
import torch
from ultralytics import YOLO

last = RUNS / NAME / "weights" / "last.pt"
finished = last.is_file() and torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1
if finished:
    print("Training already finished:", last.parent)
elif last.is_file():
    YOLO(str(last)).train(resume=True)
else:
    YOLO(MODEL).train(
        data=str(DATA_DIR / "data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=2,
        project=str(RUNS), name=NAME, exist_ok=True, seed=0, plots=True, single_cls=True,
        fliplr=0.5, degrees=0.0, scale=0.3, translate=0.1, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3,
        mosaic=1.0, close_mosaic=10,
    )
BEST = RUNS / NAME / "weights" / "best.pt"
print("best weights:", BEST)

## 5 · Validation on the held-out train clips (the real test is B04 on the laptop)

In [ ]:
m = YOLO(str(BEST)).val(data=str(DATA_DIR / "data.yaml"), imgsz=IMGSZ, batch=BATCH, plots=False)
print(f"mAP50 {m.box.map50:.3f} | precision {m.box.mp:.3f} | recall {m.box.mr:.3f}")
print("Download", BEST, "to weights/" + NAME + ".pt on the laptop")